# Multi-latent merger recovery

**Question:** Do experiments provide complementary information that improves merger recovery?

This pilot compares Run 11 experiments 9, 13, and 14 for major mergers within 0.75 Gyr. It combines normalized high-dimensional geometry and, separately, validation-calibrated UMAP retrieval scores. Existing representations and the prior choice of target make this an exploratory comparison.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import json
import sys
from IPython.display import Image, Markdown, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "multi_latent_analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the Hyrax-Research repository tree.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import multi_latent_analysis as mla
import static_umap_metrics as metrics

## Configuration

The default cells load the completed pilot. Set `RUN_ANALYSIS=True` and choose a **new output directory** to recompute. Install `multi_latent_requirements.txt` in the notebook kernel first. See `MULTI_LATENT_ANALYSIS.md` for the manifest interface and CLI.

In [ ]:
MANIFEST = ROOT / "results/multi_latent_run11/inputs_manifest.json"
OUTPUT = ROOT / "results/multi_latent_run11/full"
CONFIG = mla.AnalysisConfig()
RUN_ANALYSIS = False

if RUN_ANALYSIS:
    mla.run_analysis(MANIFEST, OUTPUT, CONFIG)
bundle = metrics.load_analysis_bundle(OUTPUT, expected_config=asdict(CONFIG))
tables = bundle["tables"]
display(Markdown((OUTPUT / "meeting_brief.md").read_text()))

## 1. Recovery and controls

Every method sees the same 50/25/25 object split within each repetition. Training positives define anchors; validation chooses the neighbor rank and threshold. Purity and completeness below use the frozen threshold on test galaxies. AP describes the full test ranking. Bands and bars show 16th–84th percentile **split variability**, not confidence intervals.

In [ ]:
display(Image(filename=str(OUTPUT / "01_recovery.png")))
columns = ["mode", "method", "average_precision", "average_precision_p16",
           "average_precision_p84", "purity", "completeness", "f1", "prevalence"]
display(tables["summary"][columns])
display(tables["paired_summary"])

## 2. Complementarity and agreement

CKA compares high-dimensional representations on a fixed unlabeled sample. Candidate overlap is the median Jaccard of the top 10% of test objects across splits. Neither diagnostic substitutes for the recovery comparison. [CKA reference: Kornblith et al. (2019)](https://proceedings.mlr.press/v97/kornblith19a.html).

In [ ]:
display(Image(filename=str(OUTPUT / "02_agreement.png")))

## 3. Spatial interpretation

The right panel is a **new, descriptive projection** of the combined latent geometry, using all-object RMS normalization, UMAP neighbors=15, min_dist=0.1, Euclidean distance, and seed 42. It is not used for retrieval evaluation. The two maps have independent coordinate systems.

In [ ]:
display(Image(filename=str(OUTPUT / "03_spatial_interpretation.png")))

## Audit and next step

The source artifacts are read-only. IDs are joined exactly; repeated filter rows do not inflate denominators. Unknown labels are excluded and reported. `split_audit.npz` preserves all object partitions and held-out scores.

Freeze the preferred method before evaluating an untouched galaxy sample or snapshot. If gains are inconsistent, test which ensemble member contributes using a leave-one-experiment-out study.

In [ ]:
display(bundle["metadata"]["coverage"])
display(bundle["metadata"]["versions"])
print("Results:", OUTPUT)
print("Source fingerprint:", bundle["metadata"]["source_sha256"])